In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
from pathlib import Path
from collections import Counter
import torch

input_root = Path("/kaggle/input")
files = [path for path in input_root.rglob("*") if path.is_file()]

print("CUDA disponibile:", torch.cuda.is_available())
print("Numero GPU:", torch.cuda.device_count())

for index in range(torch.cuda.device_count()):
    print(index, torch.cuda.get_device_name(index))

print("\nDataset collegati:")
for path in input_root.iterdir():
    print("-", path)

print("\nFile totali:", len(files))
print("Dimensione totale:", round(sum(p.stat().st_size for p in files) / 1024**3, 2), "GB")
print("Estensioni:", Counter(p.suffix.lower() for p in files))

print("\nPrimi 80 file:")
for path in files[:80]:
    print(path)

In [ ]:
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd

DATA_ROOT = Path(
    "/kaggle/input/datasets/neelu1/"
    "thermal-anomaly-detection-dataset/"
    "harbor_anomaly_dataset"
)

print("Cartelle principali:")
for path in sorted(DATA_ROOT.iterdir()):
    print("-", path.name)

# Raggruppa i JPG per clip
clips = defaultdict(list)

for frame_path in DATA_ROOT.rglob("*.jpg"):
    clips[frame_path.parent].append(frame_path)

clip_rows = []

for clip_dir, frames in clips.items():
    relative = clip_dir.relative_to(DATA_ROOT)
    clip_rows.append({
        "split": relative.parts[0],
        "date": relative.parts[-2],
        "clip": relative.parts[-1],
        "frames": len(frames),
    })

clip_table = pd.DataFrame(clip_rows).sort_values(
    ["split", "date", "clip"]
)

print("\nRiepilogo clip:")
display(
    clip_table.groupby("split")
    .agg(clips=("clip", "count"), frames=("frames", "sum"))
)

print("\nPrime clip:")
display(clip_table.head(10))

# Analizza le 29 annotazioni temporali
annotation_rows = []

for annotation_path in sorted(
    (DATA_ROOT / "test_annotations_anomalies").rglob("*.npy")
):
    labels = np.load(annotation_path, allow_pickle=False).reshape(-1)

    date = annotation_path.parent.name
    clip = annotation_path.stem
    frame_dir = DATA_ROOT / "test" / date / clip
    frame_count = len(list(frame_dir.glob("*.jpg")))

    annotation_rows.append({
        "date": date,
        "clip": clip,
        "frames_jpg": frame_count,
        "labels": len(labels),
        "normal_frames": int(np.sum(labels == 0)),
        "anomaly_frames": int(np.sum(labels != 0)),
        "values": str(np.unique(labels).tolist()),
        "length_ok": frame_count == len(labels),
    })

annotation_table = pd.DataFrame(annotation_rows)

print("\nAnnotazioni test:")
display(annotation_table)

print("\nTotali annotati:")
print(annotation_table[
    ["frames_jpg", "normal_frames", "anomaly_frames"]
].sum())

print(
    "Tutte le lunghezze coincidono:",
    annotation_table["length_ok"].all()
)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from PIL import Image
import numpy as np

WORK = Path("/kaggle/working")

clip_table.to_csv(WORK / "E07_clip_inventory.csv", index=False)
annotation_table.to_csv(
    WORK / "E07_annotation_inventory.csv",
    index=False
)

# Seleziona clip con poche, medie e molte anomalie
ordered = annotation_table.sort_values("anomaly_frames").reset_index(drop=True)
selected = ordered.iloc[[0, len(ordered) // 2, len(ordered) - 1]]

fig, axes = plt.subplots(3, 2, figsize=(12, 12))

for row_index, (_, row) in enumerate(selected.iterrows()):
    annotation_path = (
        DATA_ROOT
        / "test_annotations_anomalies"
        / str(row["date"])
        / f'{row["clip"]}.npy'
    )

    frame_dir = (
        DATA_ROOT
        / "test"
        / str(row["date"])
        / row["clip"]
    )

    labels = np.load(annotation_path, allow_pickle=False).reshape(-1)
    frames = sorted(frame_dir.glob("*.jpg"))

    normal_indices = np.where(labels == 0)[0]
    anomaly_indices = np.where(labels != 0)[0]

    normal_index = normal_indices[len(normal_indices) // 2]
    anomaly_index = anomaly_indices[len(anomaly_indices) // 2]

    axes[row_index, 0].imshow(Image.open(frames[normal_index]))
    axes[row_index, 0].set_title(
        f'{row["date"]}/{row["clip"]} — normale, frame {normal_index}'
    )

    axes[row_index, 1].imshow(Image.open(frames[anomaly_index]))
    axes[row_index, 1].set_title(
        f'{row["date"]}/{row["clip"]} — anomalo, frame {anomaly_index}'
    )

    for axis in axes[row_index]:
        axis.axis("off")

plt.suptitle("E07 — esempi del benchmark termico esterno", fontsize=16)
plt.tight_layout()
plt.savefig(
    WORK / "E07_dataset_examples.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

# Timeline delle etichette per tutti i 29 clip
timeline_rows = []
timeline_names = []

for _, row in annotation_table.iterrows():
    labels = np.load(
        DATA_ROOT
        / "test_annotations_anomalies"
        / str(row["date"])
        / f'{row["clip"]}.npy',
        allow_pickle=False,
    ).reshape(-1)

    timeline_rows.append(labels)
    timeline_names.append(f'{row["date"]}/{row["clip"]}')

timeline = np.stack(timeline_rows)

fig, ax = plt.subplots(figsize=(14, 10))
ax.imshow(
    timeline,
    aspect="auto",
    interpolation="nearest",
    cmap="RdYlGn_r",
    vmin=0,
    vmax=1,
)
ax.set_xlabel("Frame nel clip")
ax.set_ylabel("Clip")
ax.set_yticks(range(len(timeline_names)))
ax.set_yticklabels(timeline_names, fontsize=7)
ax.set_title("E07 — ground truth temporale dei 29 clip di test")
ax.legend(
    handles=[
        Patch(color="green", label="Normale"),
        Patch(color="red", label="Anomalo"),
    ],
    loc="upper right",
)
plt.tight_layout()
plt.savefig(
    WORK / "E07_ground_truth_timeline.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

print("CSV e figure E07 salvati in /kaggle/working")

In [ ]:
%cd /kaggle/working

!test -d AnomalyVFM || git clone \
  https://github.com/MaticFuc/AnomalyVFM.git

%cd /kaggle/working/AnomalyVFM

!git checkout 4da96b493a0b12e4e380fa5bf6573c126342855c

!pip install -q \
  "huggingface_hub==0.36.2" \
  "transformers==4.56.1" \
  "timm==1.0.25" \
  "einops==0.8.2" \
  safetensors

print("Installazione completata")
!git rev-parse HEAD

In [ ]:
import torch
import transformers
import huggingface_hub
import timm

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)
print("Timm:", timm.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", [torch.cuda.get_device_name(i)
               for i in range(torch.cuda.device_count())])

In [ ]:
%cd /kaggle/working/AnomalyVFM

import sys
import subprocess
from pathlib import Path
from huggingface_hub import model_info

sys.path.insert(0, "/kaggle/working/AnomalyVFM")

from hf_model import AnomalyVFM

HF_MODEL_ID = "MaticFuc/anomalyvfm_dinov2"
HF_REVISION = "c7698cefc6415ca43c93c3e924e02f6863c57e73"
DEVICE = torch.device("cuda:0")

base_model = AnomalyVFM.from_pretrained(
    HF_MODEL_ID,
    revision=HF_REVISION,
).to(DEVICE)

base_model.eval()
image_transform = base_model.model.get_img_transform()

parameter_count = sum(
    parameter.numel()
    for parameter in base_model.parameters()
)

# Usa entrambe le T4 durante i batch
if torch.cuda.device_count() >= 2:
    anomalyvfm = torch.nn.DataParallel(
        base_model,
        device_ids=[0, 1],
    )
else:
    anomalyvfm = base_model

anomalyvfm.eval()

commit = subprocess.check_output(
    [
        "git",
        "-C",
        "/kaggle/working/AnomalyVFM",
        "rev-parse",
        "HEAD",
    ],
    text=True,
).strip()

print("Modello:", HF_MODEL_ID)
print("Revisione:", HF_REVISION)
print("Commit:", commit)
print("Parametri:", parameter_count)
print("GPU utilizzate:", torch.cuda.device_count())
print("Modello caricato correttamente")

In [ ]:
from PIL import Image
import pandas as pd
import numpy as np
import time

# Manifest completo dei 3.509 frame test
manifest_rows = []

for annotation_path in sorted(
    (DATA_ROOT / "test_annotations_anomalies").rglob("*.npy")
):
    date = annotation_path.parent.name
    clip = annotation_path.stem

    labels = np.load(
        annotation_path,
        allow_pickle=False,
    ).reshape(-1)

    frame_dir = DATA_ROOT / "test" / date / clip
    frame_paths = sorted(frame_dir.glob("*.jpg"))

    for frame_index, (frame_path, label) in enumerate(
        zip(frame_paths, labels)
    ):
        manifest_rows.append({
            "date": date,
            "clip": clip,
            "frame_index": frame_index,
            "path": str(frame_path),
            "label": int(label != 0),
        })

test_manifest = pd.DataFrame(manifest_rows)

test_manifest.to_csv(
    "/kaggle/working/E07_test_manifest.csv",
    index=False,
)

print("Manifest:", test_manifest.shape)
print(test_manifest["label"].value_counts().sort_index())

# Campione bilanciato riproducibile
probe_manifest = pd.concat([
    test_manifest[test_manifest["label"] == 0].sample(
        8, random_state=7
    ),
    test_manifest[test_manifest["label"] == 1].sample(
        8, random_state=7
    ),
]).reset_index(drop=True)

probe_batch = torch.stack([
    image_transform(
        Image.open(path).convert("RGB")
    )
    for path in probe_manifest["path"]
]).to(DEVICE)

for gpu in range(torch.cuda.device_count()):
    torch.cuda.reset_peak_memory_stats(gpu)
    torch.cuda.synchronize(gpu)

start = time.time()

with torch.inference_mode():
    probe_scores, probe_masks = anomalyvfm(probe_batch)

for gpu in range(torch.cuda.device_count()):
    torch.cuda.synchronize(gpu)

elapsed = time.time() - start

probe_manifest["score"] = (
    probe_scores.detach().float().cpu().reshape(-1).numpy()
)

probe_manifest.to_csv(
    "/kaggle/working/E07_probe_results.csv",
    index=False,
)

print("Forma score:", tuple(probe_scores.shape))
print("Forma maschere:", tuple(probe_masks.shape))
print("Tempo batch 16:", round(elapsed, 3), "s")

for gpu in range(torch.cuda.device_count()):
    print(
        f"Picco GPU {gpu}:",
        round(
            torch.cuda.max_memory_allocated(gpu) / 1024**3,
            2,
        ),
        "GB",
    )

display(
    probe_manifest[
        ["date", "clip", "frame_index", "label", "score"]
    ]
)

In [ ]:
from tqdm.auto import tqdm
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)
import time
import numpy as np
import pandas as pd

BATCH_SIZE = 16
NATIVE_THRESHOLD = 0.5

# Libera il probe
del probe_batch, probe_scores, probe_masks
torch.cuda.empty_cache()

score_parts = []
mask_parts = []

for gpu in range(torch.cuda.device_count()):
    torch.cuda.reset_peak_memory_stats(gpu)
    torch.cuda.synchronize(gpu)

start_time = time.time()

for start_index in tqdm(
    range(0, len(test_manifest), BATCH_SIZE),
    desc="Inferenza E07 AnomalyVFM",
):
    batch_rows = test_manifest.iloc[
        start_index:start_index + BATCH_SIZE
    ]

    batch = torch.stack([
        image_transform(
            Image.open(path).convert("RGB")
        )
        for path in batch_rows["path"]
    ]).to(DEVICE)

    with torch.inference_mode():
        batch_scores, batch_masks = anomalyvfm(batch)

    score_parts.append(
        batch_scores.detach().float().cpu().reshape(-1).numpy()
    )
    mask_parts.append(
        batch_masks.detach().float().cpu().numpy()
    )

    del batch, batch_scores, batch_masks

for gpu in range(torch.cuda.device_count()):
    torch.cuda.synchronize(gpu)

inference_seconds = time.time() - start_time

scores = np.concatenate(score_parts)
raw_masks = np.concatenate(mask_parts).squeeze(1)
labels = test_manifest["label"].to_numpy()

results = test_manifest.copy()
results["score"] = scores
results["prediction_native_0_5"] = (
    scores >= NATIVE_THRESHOLD
).astype(int)

results.to_csv(
    "/kaggle/working/E07_scores.csv",
    index=False,
)

# NPZ portabile senza pickle
np.savez_compressed(
    "/kaggle/working/E07_raw_masks_float16.npz",
    masks=raw_masks.astype(np.float16),
    scores=scores.astype(np.float32),
    labels=labels.astype(np.int8),
    dates=results["date"].astype(str).to_numpy(dtype="U8"),
    clips=results["clip"].astype(str).to_numpy(dtype="U32"),
    frame_indices=results["frame_index"].to_numpy(dtype=np.int16),
)

predictions = results["prediction_native_0_5"].to_numpy()

tn, fp, fn, tp = confusion_matrix(
    labels,
    predictions,
    labels=[0, 1],
).ravel()

classification_summary = pd.DataFrame([{
    "experiment_id": "E07_v1_anomalyvfm_external_zero_shot",
    "threshold_source": "native_fixed_0.5",
    "threshold": NATIVE_THRESHOLD,
    "normal_frames": int((labels == 0).sum()),
    "anomaly_frames": int((labels == 1).sum()),
    "tn": int(tn),
    "fp": int(fp),
    "fn": int(fn),
    "tp": int(tp),
    "precision": precision_score(
        labels, predictions, zero_division=0
    ),
    "recall": recall_score(
        labels, predictions, zero_division=0
    ),
    "f1": f1_score(
        labels, predictions, zero_division=0
    ),
    "auroc": roc_auc_score(labels, scores),
    "average_precision": average_precision_score(
        labels, scores
    ),
    "inference_seconds": inference_seconds,
    "frames_per_second": len(results) / inference_seconds,
}])

classification_summary.to_csv(
    "/kaggle/working/E07_classification_native_summary.csv",
    index=False,
)

score_statistics = (
    results.assign(
        role=np.where(
            results["label"] == 1,
            "anomaly",
            "normal",
        )
    )
    .groupby("role")["score"]
    .agg(["count", "min", "median", "mean", "max", "std"])
)

score_statistics.to_csv(
    "/kaggle/working/E07_score_statistics.csv"
)

print("Forma mappe:", raw_masks.shape)
print("Tempo:", round(inference_seconds, 2), "s")
print("FPS:", round(len(results) / inference_seconds, 3))

for gpu in range(torch.cuda.device_count()):
    print(
        f"Picco GPU {gpu}:",
        round(
            torch.cuda.max_memory_allocated(gpu) / 1024**3,
            2,
        ),
        "GB",
    )

print("\nStatistiche score:")
display(score_statistics)

print("\nMetriche soglia nativa 0.5:")
display(classification_summary.T)

In [ ]:
print("STATISTICHE SCORE")
print(score_statistics.to_string())

print("\nMETRICHE SOGLIA NATIVA 0.5")
print(classification_summary.T.to_string(header=False))

In [ ]:
from pathlib import Path
from tqdm.auto import tqdm

# Un frame centrale per ciascuno dei 300 clip normali
train_clip_dirs = sorted({
    frame_path.parent
    for frame_path in (DATA_ROOT / "train").rglob("*.jpg")
})

calibration_rows = []

for clip_dir in train_clip_dirs:
    frames = sorted(clip_dir.glob("*.jpg"))
    selected_frame = frames[len(frames) // 2]
    relative = clip_dir.relative_to(DATA_ROOT / "train")

    calibration_rows.append({
        "date": relative.parts[-2],
        "clip": relative.parts[-1],
        "path": str(selected_frame),
    })

calibration_manifest = pd.DataFrame(calibration_rows)

print("Clip normali di calibrazione:", len(calibration_manifest))

calibration_score_parts = []
calibration_start = time.time()

for start_index in tqdm(
    range(0, len(calibration_manifest), BATCH_SIZE),
    desc="Calibrazione normale E07",
):
    batch_rows = calibration_manifest.iloc[
        start_index:start_index + BATCH_SIZE
    ]

    batch = torch.stack([
        image_transform(
            Image.open(path).convert("RGB")
        )
        for path in batch_rows["path"]
    ]).to(DEVICE)

    with torch.inference_mode():
        batch_scores, _ = anomalyvfm(batch)

    calibration_score_parts.append(
        batch_scores.detach().float().cpu().reshape(-1).numpy()
    )

    del batch, batch_scores

for gpu in range(torch.cuda.device_count()):
    torch.cuda.synchronize(gpu)

calibration_seconds = time.time() - calibration_start
calibration_scores = np.concatenate(calibration_score_parts)

calibration_manifest["score"] = calibration_scores
calibration_manifest.to_csv(
    "/kaggle/working/E07_normal_calibration_scores.csv",
    index=False,
)

CALIBRATED_THRESHOLD = float(
    np.quantile(calibration_scores, 0.95)
)

calibrated_predictions = (
    scores >= CALIBRATED_THRESHOLD
).astype(int)

tn_c, fp_c, fn_c, tp_c = confusion_matrix(
    labels,
    calibrated_predictions,
    labels=[0, 1],
).ravel()

calibrated_summary = pd.DataFrame([{
    "experiment_id": "E07_v1_anomalyvfm_external_normal_calibrated",
    "threshold_source": "95th_percentile_300_normal_train_clips",
    "threshold": CALIBRATED_THRESHOLD,
    "normal_frames": int((labels == 0).sum()),
    "anomaly_frames": int((labels == 1).sum()),
    "tn": int(tn_c),
    "fp": int(fp_c),
    "fn": int(fn_c),
    "tp": int(tp_c),
    "precision": precision_score(
        labels, calibrated_predictions, zero_division=0
    ),
    "recall": recall_score(
        labels, calibrated_predictions, zero_division=0
    ),
    "f1": f1_score(
        labels, calibrated_predictions, zero_division=0
    ),
    "auroc": roc_auc_score(labels, scores),
    "average_precision": average_precision_score(
        labels, scores
    ),
    "calibration_frames": len(calibration_manifest),
    "calibration_seconds": calibration_seconds,
}])

classification_comparison = pd.concat(
    [classification_summary, calibrated_summary],
    ignore_index=True,
)

classification_comparison.to_csv(
    "/kaggle/working/E07_classification_comparison.csv",
    index=False,
)

results["prediction_normal_calibrated"] = (
    calibrated_predictions
)
results.to_csv(
    "/kaggle/working/E07_scores.csv",
    index=False,
)

print("Soglia calibrata:", CALIBRATED_THRESHOLD)
print("Tempo calibrazione:", round(calibration_seconds, 2), "s")
display(calibrated_summary.T)

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np
import pandas as pd

results = results.sort_values(
    ["date", "clip", "frame_index"]
).reset_index(drop=True)

# Smoothing causale rispetto alle etichette: usa solo gli score
for window in [3, 5, 11]:
    results[f"score_smooth_{window}"] = (
        results.groupby(["date", "clip"])["score"]
        .transform(
            lambda series: series.rolling(
                window=window,
                center=True,
                min_periods=1,
            ).mean()
        )
    )

score_columns = {
    "raw": "score",
    "smooth_3": "score_smooth_3",
    "smooth_5": "score_smooth_5",
    "smooth_11": "score_smooth_11",
}

per_clip_rows = []
summary_rows = []

for method, score_column in score_columns.items():
    clip_aurocs = []
    clip_aps = []
    clip_deltas = []

    for (date, clip), group in results.groupby(
        ["date", "clip"]
    ):
        y_true = group["label"].to_numpy()
        clip_scores = group[score_column].to_numpy()

        clip_auroc = roc_auc_score(y_true, clip_scores)
        clip_ap = average_precision_score(
            y_true, clip_scores
        )

        normal_mean = group.loc[
            group["label"] == 0, score_column
        ].mean()

        anomaly_mean = group.loc[
            group["label"] == 1, score_column
        ].mean()

        delta = anomaly_mean - normal_mean

        clip_aurocs.append(clip_auroc)
        clip_aps.append(clip_ap)
        clip_deltas.append(delta)

        per_clip_rows.append({
            "method": method,
            "date": date,
            "clip": clip,
            "normal_frames": int((y_true == 0).sum()),
            "anomaly_frames": int((y_true == 1).sum()),
            "auroc": clip_auroc,
            "average_precision": clip_ap,
            "normal_mean_score": normal_mean,
            "anomaly_mean_score": anomaly_mean,
            "anomaly_minus_normal": delta,
        })

    summary_rows.append({
        "method": method,
        "global_auroc": roc_auc_score(
            results["label"],
            results[score_column],
        ),
        "global_average_precision": average_precision_score(
            results["label"],
            results[score_column],
        ),
        "macro_clip_auroc": np.mean(clip_aurocs),
        "median_clip_auroc": np.median(clip_aurocs),
        "macro_clip_ap": np.mean(clip_aps),
        "clips_auroc_above_0_5": int(
            np.sum(np.array(clip_aurocs) > 0.5)
        ),
        "clips_positive_score_delta": int(
            np.sum(np.array(clip_deltas) > 0)
        ),
    })

per_clip_metrics = pd.DataFrame(per_clip_rows)
temporal_summary = pd.DataFrame(summary_rows)

per_clip_metrics.to_csv(
    "/kaggle/working/E07_per_clip_metrics.csv",
    index=False,
)

temporal_summary.to_csv(
    "/kaggle/working/E07_temporal_smoothing_summary.csv",
    index=False,
)

results.to_csv(
    "/kaggle/working/E07_scores.csv",
    index=False,
)

display(temporal_summary)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_curve,
    precision_recall_curve,
)
import numpy as np

y_true = results["label"].to_numpy()
raw_scores = results["score"].to_numpy()

fpr, tpr, _ = roc_curve(y_true, raw_scores)
precision_curve, recall_curve, _ = precision_recall_curve(
    y_true, raw_scores
)

prevalence = y_true.mean()

raw_per_clip = (
    per_clip_metrics[
        per_clip_metrics["method"] == "raw"
    ]
    .sort_values("auroc")
    .reset_index(drop=True)
)

fig, axes = plt.subplots(2, 2, figsize=(15, 11))

# Distribuzione score
axes[0, 0].hist(
    results.loc[results["label"] == 0, "score"],
    bins=45,
    alpha=0.65,
    density=True,
    label="Normali",
    color="seagreen",
)
axes[0, 0].hist(
    results.loc[results["label"] == 1, "score"],
    bins=45,
    alpha=0.65,
    density=True,
    label="Anomali",
    color="darkorange",
)
axes[0, 0].axvline(
    0.5,
    color="red",
    linestyle="--",
    label="Soglia nativa 0,5",
)
axes[0, 0].axvline(
    CALIBRATED_THRESHOLD,
    color="blue",
    linestyle=":",
    label=f"Soglia calibrata {CALIBRATED_THRESHOLD:.3f}",
)
axes[0, 0].set_title("Distribuzione degli anomaly score")
axes[0, 0].set_xlabel("Score")
axes[0, 0].set_ylabel("Densità")
axes[0, 0].legend()

# ROC
axes[0, 1].plot(
    fpr,
    tpr,
    label=f"AnomalyVFM — AUROC {roc_auc_score(y_true, raw_scores):.3f}",
)
axes[0, 1].plot(
    [0, 1],
    [0, 1],
    "k--",
    label="Classificatore casuale",
)
axes[0, 1].set_title("Curva ROC")
axes[0, 1].set_xlabel("False Positive Rate")
axes[0, 1].set_ylabel("True Positive Rate")
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.25)

# Precision-Recall
axes[1, 0].plot(
    recall_curve,
    precision_curve,
    color="purple",
    label=f"AP {average_precision_score(y_true, raw_scores):.3f}",
)
axes[1, 0].axhline(
    prevalence,
    color="black",
    linestyle="--",
    label=f"Prevalenza {prevalence:.3f}",
)
axes[1, 0].set_title("Curva Precision-Recall")
axes[1, 0].set_xlabel("Recall")
axes[1, 0].set_ylabel("Precision")
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.25)

# AUROC per clip
colors = np.where(
    raw_per_clip["auroc"] >= 0.5,
    "seagreen",
    "indianred",
)
axes[1, 1].bar(
    range(len(raw_per_clip)),
    raw_per_clip["auroc"],
    color=colors,
)
axes[1, 1].axhline(
    0.5,
    color="black",
    linestyle="--",
    label="Caso casuale",
)
axes[1, 1].set_title("AUROC per clip, ordinata")
axes[1, 1].set_xlabel("Clip")
axes[1, 1].set_ylabel("AUROC")
axes[1, 1].set_ylim(0, 1)
axes[1, 1].legend()
axes[1, 1].grid(axis="y", alpha=0.25)

plt.suptitle(
    "E07 — AnomalyVFM sul Thermal Anomaly Detection Dataset",
    fontsize=16,
)
plt.tight_layout()
plt.savefig(
    "/kaggle/working/E07_quantitative_overview.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

print("Clip peggiori:")
display(raw_per_clip.head(5))

print("Clip migliori:")
display(raw_per_clip.tail(5))

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Timeline clip peggiore e migliore
worst_clip = raw_per_clip.iloc[0]
best_clip = raw_per_clip.iloc[-1]

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=False)

for axis, row, label_name in [
    (axes[0], worst_clip, "Clip peggiore"),
    (axes[1], best_clip, "Clip migliore"),
]:
    group = results[
        (results["date"].astype(str) == str(row["date"]))
        & (results["clip"] == row["clip"])
    ].sort_values("frame_index")

    x = group["frame_index"].to_numpy()

    axis.plot(
        x,
        group["score"],
        color="navy",
        label="Anomaly score",
    )
    axis.axhline(
        0.5,
        color="red",
        linestyle="--",
        label="Soglia nativa",
    )
    axis.axhline(
        CALIBRATED_THRESHOLD,
        color="blue",
        linestyle=":",
        label="Soglia calibrata",
    )
    axis.fill_between(
        x,
        0,
        1,
        where=group["label"].to_numpy() == 1,
        transform=axis.get_xaxis_transform(),
        color="orange",
        alpha=0.22,
        label="Ground truth anomala",
    )

    axis.set_title(
        f'{label_name}: {row["date"]}/{row["clip"]} '
        f'— AUROC {row["auroc"]:.3f}'
    )
    axis.set_xlabel("Frame")
    axis.set_ylabel("Score")
    axis.grid(alpha=0.25)
    axis.legend(loc="upper right")

plt.tight_layout()
plt.savefig(
    "/kaggle/working/E07_best_worst_timelines.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

# Tre normali con score più alto e tre anomalie con score più basso
high_normal = (
    results[results["label"] == 0]
    .nlargest(3, "score")
    .copy()
)
high_normal["error_type"] = "Falso positivo"

low_anomaly = (
    results[results["label"] == 1]
    .nsmallest(3, "score")
    .copy()
)
low_anomaly["error_type"] = "Falso negativo"

failure_cases = pd.concat(
    [high_normal, low_anomaly],
    ignore_index=True,
)

failure_cases.to_csv(
    "/kaggle/working/E07_failure_cases.csv",
    index=False,
)

# Recupera le mappe salvate
with np.load(
    "/kaggle/working/E07_raw_masks_float16.npz",
    allow_pickle=False,
) as archive:
    saved_masks = archive["masks"].astype(np.float32)
    saved_dates = archive["dates"].astype(str)
    saved_clips = archive["clips"].astype(str)
    saved_indices = archive["frame_indices"]

mask_lookup = {
    (date, clip, int(index)): mask
    for date, clip, index, mask in zip(
        saved_dates,
        saved_clips,
        saved_indices,
        saved_masks,
    )
}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for axis, (_, row) in zip(axes.ravel(), failure_cases.iterrows()):
    image = Image.open(row["path"]).convert("RGB")

    key = (
        str(row["date"]),
        row["clip"],
        int(row["frame_index"]),
    )
    mask = mask_lookup[key]

    low, high = np.percentile(mask, [1, 99])
    normalized = np.clip(
        (mask - low) / max(high - low, 1e-8),
        0,
        1,
    )

    resized_mask = np.asarray(
        Image.fromarray(normalized.astype(np.float32)).resize(
            image.size,
            Image.Resampling.BICUBIC,
        )
    )

    axis.imshow(image)
    axis.imshow(
        resized_mask,
        cmap="inferno",
        alpha=0.48,
        vmin=0,
        vmax=1,
    )
    axis.set_title(
        f'{row["error_type"]}\n'
        f'{row["date"]}/{row["clip"]}, '
        f'frame {row["frame_index"]}, score {row["score"]:.3f}'
    )
    axis.axis("off")

plt.suptitle(
    "E07 — mappe normalizzate dei principali errori",
    fontsize=16,
)
plt.tight_layout()
plt.savefig(
    "/kaggle/working/E07_qualitative_failure_maps.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
)
import numpy as np
import pandas as pd

def binary_segments(values):
    values = np.asarray(values, dtype=np.int8)
    padded = np.pad(values, (1, 1))
    differences = np.diff(padded)

    starts = np.where(differences == 1)[0]
    ends = np.where(differences == -1)[0] - 1

    return list(zip(starts.tolist(), ends.tolist()))

def temporal_iou(first, second):
    intersection_start = max(first[0], second[0])
    intersection_end = min(first[1], second[1])

    intersection = max(
        0,
        intersection_end - intersection_start + 1,
    )

    first_length = first[1] - first[0] + 1
    second_length = second[1] - second[0] + 1
    union = first_length + second_length - intersection

    return intersection / union if union else 0.0

def greedy_temporal_matches(gt_segments, predicted_segments, threshold):
    candidates = []

    for gt_index, gt_segment in enumerate(gt_segments):
        for prediction_index, prediction_segment in enumerate(
            predicted_segments
        ):
            value = temporal_iou(gt_segment, prediction_segment)

            if value > 0 and value >= threshold:
                candidates.append(
                    (value, gt_index, prediction_index)
                )

    candidates.sort(reverse=True)

    matched_gt = set()
    matched_predictions = set()
    matches = []

    for value, gt_index, prediction_index in candidates:
        if (
            gt_index not in matched_gt
            and prediction_index not in matched_predictions
        ):
            matched_gt.add(gt_index)
            matched_predictions.add(prediction_index)
            matches.append(value)

    return matches

temporal_methods = {
    "native_raw": ("score", 0.5),
    "normal_calibrated_raw": (
        "score",
        CALIBRATED_THRESHOLD,
    ),
    "normal_calibrated_smooth11": (
        "score_smooth_11",
        CALIBRATED_THRESHOLD,
    ),
}

event_rows = []
segment_rows = []

for method, (score_column, threshold) in temporal_methods.items():
    frame_predictions = (
        results[score_column].to_numpy() >= threshold
    ).astype(int)

    for temporal_threshold in [0.1, 0.3, 0.5]:
        total_tp = 0
        total_fp = 0
        total_fn = 0
        matched_ious = []

        for (date, clip), group in results.groupby(
            ["date", "clip"]
        ):
            group = group.sort_values("frame_index")

            gt_segments = binary_segments(
                group["label"].to_numpy()
            )
            predicted_segments = binary_segments(
                (
                    group[score_column].to_numpy()
                    >= threshold
                ).astype(int)
            )

            matches = greedy_temporal_matches(
                gt_segments,
                predicted_segments,
                temporal_threshold,
            )

            total_tp += len(matches)
            total_fp += len(predicted_segments) - len(matches)
            total_fn += len(gt_segments) - len(matches)
            matched_ious.extend(matches)

            if temporal_threshold == 0.1:
                segment_rows.append({
                    "method": method,
                    "date": date,
                    "clip": clip,
                    "gt_events": len(gt_segments),
                    "predicted_events": len(predicted_segments),
                    "matched_events_tiou_0_1": len(matches),
                    "best_tiou": max(matches) if matches else 0.0,
                })

        event_precision = (
            total_tp / (total_tp + total_fp)
            if total_tp + total_fp else 0.0
        )
        event_recall = (
            total_tp / (total_tp + total_fn)
            if total_tp + total_fn else 0.0
        )
        event_f1 = (
            2 * event_precision * event_recall
            / (event_precision + event_recall)
            if event_precision + event_recall else 0.0
        )

        event_rows.append({
            "method": method,
            "temporal_iou_threshold": temporal_threshold,
            "tp_events": total_tp,
            "fp_events": total_fp,
            "fn_events": total_fn,
            "event_precision": event_precision,
            "event_recall": event_recall,
            "event_f1": event_f1,
            "mean_matched_tiou": (
                np.mean(matched_ious)
                if matched_ious else 0.0
            ),
            "frame_precision": precision_score(
                results["label"],
                frame_predictions,
                zero_division=0,
            ),
            "frame_recall": recall_score(
                results["label"],
                frame_predictions,
                zero_division=0,
            ),
            "frame_f1": f1_score(
                results["label"],
                frame_predictions,
                zero_division=0,
            ),
        })

temporal_event_summary = pd.DataFrame(event_rows)
temporal_segment_details = pd.DataFrame(segment_rows)

temporal_event_summary.to_csv(
    "/kaggle/working/E07_temporal_event_summary.csv",
    index=False,
)
temporal_segment_details.to_csv(
    "/kaggle/working/E07_temporal_segment_details.csv",
    index=False,
)

display(temporal_event_summary)

In [ ]:
import numpy as np
import pandas as pd

results = pd.read_csv(
    "/kaggle/working/E07_scores.csv"
)

calibration_scores_table = pd.read_csv(
    "/kaggle/working/E07_normal_calibration_scores.csv"
)

CALIBRATED_THRESHOLD = float(
    np.quantile(
        calibration_scores_table["score"].to_numpy(),
        0.95,
    )
)

# Ricrea lo smoothing se necessario
if "score_smooth_11" not in results.columns:
    results = results.sort_values(
        ["date", "clip", "frame_index"]
    ).reset_index(drop=True)

    for window in [3, 5, 11]:
        results[f"score_smooth_{window}"] = (
            results.groupby(["date", "clip"])["score"]
            .transform(
                lambda series: series.rolling(
                    window,
                    center=True,
                    min_periods=1,
                ).mean()
            )
        )

print("Soglia recuperata:", CALIBRATED_THRESHOLD)
print("Colonne score:", [
    column for column in results.columns
    if column.startswith("score")
])

In [ ]:
print("Variabile results:", "results" in globals())
print("Variabile raw_masks:", "raw_masks" in globals())
print("Variabile anomalyvfm:", "anomalyvfm" in globals())

!find /kaggle/working -maxdepth 2 -type f | head -30

In [ ]:
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)
from IPython.display import FileLink, display
import hashlib
import json
import shutil
import time
import zipfile
import numpy as np
import pandas as pd

DATA_ROOT = Path(
    "/kaggle/input/datasets/neelu1/"
    "thermal-anomaly-detection-dataset/"
    "harbor_anomaly_dataset"
)
WORK = Path("/kaggle/working")

# Ricostruzione manifest dei 3.509 frame
manifest_rows = []

for annotation_path in sorted(
    (DATA_ROOT / "test_annotations_anomalies").rglob("*.npy")
):
    date = annotation_path.parent.name
    clip = annotation_path.stem

    labels_clip = np.load(
        annotation_path,
        allow_pickle=False,
    ).reshape(-1)

    frame_paths = sorted(
        (DATA_ROOT / "test" / date / clip).glob("*.jpg")
    )

    for frame_index, (frame_path, label) in enumerate(
        zip(frame_paths, labels_clip)
    ):
        manifest_rows.append({
            "date": date,
            "clip": clip,
            "frame_index": frame_index,
            "path": str(frame_path),
            "label": int(label != 0),
        })

test_manifest = pd.DataFrame(manifest_rows)
test_manifest.to_csv(
    WORK / "E07_test_manifest.csv",
    index=False,
)

print("Manifest:", test_manifest.shape)
print(test_manifest["label"].value_counts().sort_index())

# Verifica automatica se batch 24 entra in memoria
BATCH_SIZE = 24

try:
    test_batch = torch.stack([
        image_transform(
            Image.open(path).convert("RGB")
        )
        for path in test_manifest.iloc[:BATCH_SIZE]["path"]
    ]).to(DEVICE)

    with torch.inference_mode():
        test_scores, test_masks = anomalyvfm(test_batch)

    del test_batch, test_scores, test_masks
    torch.cuda.empty_cache()
    print("Batch 24 supportato")

except RuntimeError as error:
    if "out of memory" not in str(error).lower():
        raise

    BATCH_SIZE = 16
    torch.cuda.empty_cache()
    print("Batch 24 troppo grande: uso batch 16")

# Inferenza completa
score_parts = []
mask_parts = []

for gpu in range(torch.cuda.device_count()):
    torch.cuda.reset_peak_memory_stats(gpu)
    torch.cuda.synchronize(gpu)

start_time = time.time()

for start_index in tqdm(
    range(0, len(test_manifest), BATCH_SIZE),
    desc=f"Recupero E07 — batch {BATCH_SIZE}",
):
    batch_rows = test_manifest.iloc[
        start_index:start_index + BATCH_SIZE
    ]

    batch = torch.stack([
        image_transform(
            Image.open(path).convert("RGB")
        )
        for path in batch_rows["path"]
    ]).to(DEVICE)

    with torch.inference_mode():
        batch_scores, batch_masks = anomalyvfm(batch)

    score_parts.append(
        batch_scores.detach().float().cpu().reshape(-1).numpy()
    )
    mask_parts.append(
        batch_masks.detach().float().cpu().numpy()
    )

    del batch, batch_scores, batch_masks

for gpu in range(torch.cuda.device_count()):
    torch.cuda.synchronize(gpu)

inference_seconds = time.time() - start_time

scores = np.concatenate(score_parts)
raw_masks = np.concatenate(mask_parts).squeeze(1)
labels = test_manifest["label"].to_numpy()

results = test_manifest.copy()
results["score"] = scores
results["prediction_native_0_5"] = (
    scores >= 0.5
).astype(np.int8)

results.to_csv(
    WORK / "E07_scores.csv",
    index=False,
)

np.savez_compressed(
    WORK / "E07_raw_masks_float16.npz",
    masks=raw_masks.astype(np.float16),
    scores=scores.astype(np.float32),
    labels=labels.astype(np.int8),
    dates=np.array(results["date"].astype(str), dtype="U8"),
    clips=np.array(results["clip"].astype(str), dtype="U32"),
    frame_indices=results["frame_index"].to_numpy(
        dtype=np.int16
    ),
)

predictions = results["prediction_native_0_5"].to_numpy()

tn, fp, fn, tp = confusion_matrix(
    labels,
    predictions,
    labels=[0, 1],
).ravel()

classification_summary = pd.DataFrame([{
    "experiment_id": "E07_v1_anomalyvfm_external_zero_shot",
    "threshold_source": "native_fixed_0.5",
    "threshold": 0.5,
    "batch_size": BATCH_SIZE,
    "normal_frames": int((labels == 0).sum()),
    "anomaly_frames": int((labels == 1).sum()),
    "tn": int(tn),
    "fp": int(fp),
    "fn": int(fn),
    "tp": int(tp),
    "precision": precision_score(
        labels, predictions, zero_division=0
    ),
    "recall": recall_score(
        labels, predictions, zero_division=0
    ),
    "f1": f1_score(
        labels, predictions, zero_division=0
    ),
    "auroc": roc_auc_score(labels, scores),
    "average_precision": average_precision_score(
        labels, scores
    ),
    "inference_seconds": inference_seconds,
    "frames_per_second": len(results) / inference_seconds,
}])

classification_summary.to_csv(
    WORK / "E07_classification_native_summary.csv",
    index=False,
)

score_statistics = (
    results.assign(
        role=np.where(
            results["label"] == 1,
            "anomaly",
            "normal",
        )
    )
    .groupby("role")["score"]
    .agg(["count", "min", "median", "mean", "max", "std"])
)

score_statistics.to_csv(
    WORK / "E07_score_statistics.csv"
)

# Metadata del checkpoint
metadata = {
    "experiment_id":
        "E07_v1_anomalyvfm_external_zero_shot",
    "dataset":
        "neelu1/thermal-anomaly-detection-dataset",
    "train_clips": 300,
    "test_clips": 29,
    "test_frames": 3509,
    "normal_test_frames": 2385,
    "anomaly_test_frames": 1124,
    "model": "MaticFuc/anomalyvfm_dinov2",
    "model_revision":
        "c7698cefc6415ca43c93c3e924e02f6863c57e73",
    "repository_commit":
        "4da96b493a0b12e4e380fa5bf6573c126342855c",
    "batch_size": BATCH_SIZE,
    "gpus": [
        torch.cuda.get_device_name(index)
        for index in range(torch.cuda.device_count())
    ],
    "inference_seconds": inference_seconds,
    "frames_per_second":
        len(results) / inference_seconds,
}

with open(
    WORK / "E07_base_metadata.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(metadata, file, indent=2)

# ZIP immediato di sicurezza
bundle = WORK / "E07_base_checkpoint"

if bundle.exists():
    shutil.rmtree(bundle)
bundle.mkdir()

base_files = [
    "E07_test_manifest.csv",
    "E07_scores.csv",
    "E07_raw_masks_float16.npz",
    "E07_classification_native_summary.csv",
    "E07_score_statistics.csv",
    "E07_base_metadata.json",
]

for filename in base_files:
    shutil.copy2(WORK / filename, bundle / filename)

zip_path = WORK / "E07_base_checkpoint.zip"

if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED,
) as archive:
    for path in sorted(bundle.iterdir()):
        archive.write(
            path,
            arcname=f"E07_base_checkpoint/{path.name}",
        )

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)
    return digest.hexdigest()

print("\nStatistiche:")
display(score_statistics)

print("\nMetriche:")
display(classification_summary.T)

print("\nZIP creato:", zip_path)
print("SHA-256:", sha256_file(zip_path))
display(FileLink(str(zip_path)))

In [ ]:
from IPython.display import FileLink, display
from tqdm.auto import tqdm
from PIL import Image
import hashlib
import time
import zipfile
import numpy as np
import pandas as pd

# Un frame centrale per ciascuno dei 300 clip normali
train_clip_dirs = sorted({
    path.parent
    for path in (DATA_ROOT / "train").rglob("*.jpg")
})

calibration_rows = []

for clip_dir in train_clip_dirs:
    frames = sorted(clip_dir.glob("*.jpg"))
    selected = frames[len(frames) // 2]
    relative = clip_dir.relative_to(DATA_ROOT / "train")

    calibration_rows.append({
        "date": relative.parts[-2],
        "clip": relative.parts[-1],
        "path": str(selected),
    })

calibration_manifest = pd.DataFrame(calibration_rows)
calibration_parts = []

calibration_start = time.time()

for start_index in tqdm(
    range(0, len(calibration_manifest), BATCH_SIZE),
    desc="Calibrazione normale E07",
):
    rows = calibration_manifest.iloc[
        start_index:start_index + BATCH_SIZE
    ]

    batch = torch.stack([
        image_transform(
            Image.open(path).convert("RGB")
        )
        for path in rows["path"]
    ]).to(DEVICE)

    with torch.inference_mode():
        batch_scores, batch_masks = anomalyvfm(batch)

    calibration_parts.append(
        batch_scores.detach().float().cpu().reshape(-1).numpy()
    )

    del batch, batch_scores, batch_masks

for gpu in range(torch.cuda.device_count()):
    torch.cuda.synchronize(gpu)

calibration_seconds = time.time() - calibration_start
calibration_scores = np.concatenate(calibration_parts)

calibration_manifest["score"] = calibration_scores
calibration_manifest.to_csv(
    WORK / "E07_normal_calibration_scores.csv",
    index=False,
)

CALIBRATED_THRESHOLD = float(
    np.quantile(calibration_scores, 0.95)
)

labels = results["label"].to_numpy()
scores = results["score"].to_numpy()

calibrated_predictions = (
    scores >= CALIBRATED_THRESHOLD
).astype(np.int8)

tn_c, fp_c, fn_c, tp_c = confusion_matrix(
    labels,
    calibrated_predictions,
    labels=[0, 1],
).ravel()

calibrated_summary = pd.DataFrame([{
    "experiment_id":
        "E07_v1_anomalyvfm_external_normal_calibrated",
    "threshold_source":
        "95th_percentile_300_normal_train_clips",
    "threshold": CALIBRATED_THRESHOLD,
    "normal_frames": int((labels == 0).sum()),
    "anomaly_frames": int((labels == 1).sum()),
    "tn": int(tn_c),
    "fp": int(fp_c),
    "fn": int(fn_c),
    "tp": int(tp_c),
    "precision": precision_score(
        labels, calibrated_predictions, zero_division=0
    ),
    "recall": recall_score(
        labels, calibrated_predictions, zero_division=0
    ),
    "f1": f1_score(
        labels, calibrated_predictions, zero_division=0
    ),
    "auroc": roc_auc_score(labels, scores),
    "average_precision": average_precision_score(
        labels, scores
    ),
    "calibration_frames": len(calibration_manifest),
    "calibration_seconds": calibration_seconds,
}])

classification_comparison = pd.concat(
    [classification_summary, calibrated_summary],
    ignore_index=True,
)

classification_comparison.to_csv(
    WORK / "E07_classification_comparison.csv",
    index=False,
)

results["prediction_normal_calibrated"] = (
    calibrated_predictions
)
results.to_csv(WORK / "E07_scores.csv", index=False)

# ZIP di sicurezza della calibrazione
calibration_zip = WORK / "E07_calibration_checkpoint.zip"

with zipfile.ZipFile(
    calibration_zip,
    "w",
    zipfile.ZIP_DEFLATED,
) as archive:
    for filename in [
        "E07_normal_calibration_scores.csv",
        "E07_classification_comparison.csv",
        "E07_scores.csv",
    ]:
        archive.write(
            WORK / filename,
            arcname=f"E07_calibration_checkpoint/{filename}",
        )

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)
    return digest.hexdigest()

print("Soglia calibrata:", CALIBRATED_THRESHOLD)
print("Tempo:", round(calibration_seconds, 2), "s")
display(calibrated_summary.T)

print("SHA-256:", sha256_file(calibration_zip))
display(FileLink(str(calibration_zip)))

In [ ]:
from IPython.display import HTML, display
from pathlib import Path

file_path = Path(
    "/kaggle/working/E07_calibration_checkpoint.zip"
)

print(
    "File presente:",
    file_path.exists(),
    "Dimensione:",
    file_path.stat().st_size if file_path.exists() else 0,
)

display(HTML("""
<a href="/kaggle/working/E07_calibration_checkpoint.zip"
   download="E07_calibration_checkpoint.zip"
   style="
       display:inline-block;
       padding:14px 22px;
       background:#1976d2;
       color:white;
       font-weight:bold;
       text-decoration:none;
       border-radius:6px;">
   SCARICA E07 CALIBRATION ZIP
</a>
"""))